In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans 'Contract_Amount' (removes $ and , symbols) and 
      casts to Double. Numerator identifies contracts >= 1,000,000.
   3. COMMA EXCEPTION HANDLING: Protects unit names containing commas (e.g., 'CAD PB - 
      RESL, Everyday Banking...') using a '[COMMA]' placeholder swap. This prevents 
      the SQL split/explode function from breaking a single unit name into multiple parts.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists in Col K/L. Restores the '[COMMA]' to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax 
      ('%' || String || '%') to avoid regex failures on special characters.
   6. DUAL-BRIDGE JOIN: Crucial fix for mixed-format mapping data. Bridges the 
      mapping table to the Master list on EITHER AU_Name OR Numeric BusinessID.
   7. AGGREGATING: Calculates Percentage (Numerator/Denominator * 100).
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings
    -- Accommodates Mapping Table's mix of Numeric IDs and String Names
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract, Clean Amount, and Apply Comma Protection Dictionary
    SELECT 
        EngagementNumber,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    -- 4. Explode comma-separated strings and Restore the protected commas
    SELECT 
        EngagementNumber, 
        Cleaned_Amount,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5. Map expanded rows to AU using Dual-Bridge and calculate Num/Denom
    SELECT 
        mast.BusinessID,
        -- Denominator: Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT f.EngagementNumber) AS Denominator,
        -- Numerator: Distinct Engagements >= $1MM
        COUNT(DISTINCT CASE WHEN f.Cleaned_Amount >= 1000000 THEN f.EngagementNumber END) AS Numerator
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
    -- DUAL-BRIDGE JOIN: Matches on either String Name OR Numeric BusinessID
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 6. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Final Percentage Math: Handles division by zero and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - Traceability Verification
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    SELECT 
        p.EngagementNumber,
        p.Contract_Amount AS Raw_String_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(p.Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Numeric_Amount,
        p.owning_lob AS Raw_Col_K,
        
        -- Exception Protection Trace
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS protected_string,
            
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Restored_Chunk
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    LATERAL VIEW explode(split(concat_ws(',', p.owning_lob, p.lob_using), ',')) AS exploded_lob
)

SELECT DISTINCT
    f.EngagementNumber,
    f.Raw_String_Amount,
    f.Cleaned_Numeric_Amount,
    CASE WHEN f.Cleaned_Numeric_Amount >= 1000000 THEN 'YES (>= $1M)' ELSE 'NO (< $1M)' END AS Numerator_Eligibility,
    f.Raw_Col_K AS Original_LOB_String,
    f.Restored_Chunk AS Individual_Mapped_LOB,
    map.TPRM_Units AS Matched_In_Mapping_Table,
    map.Assessable_Unit_ID AS Mapping_Table_Bridge_Value,
    mast.Master_Numeric_ID AS Successfully_Bridged_To_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Bridge_Status
FROM Flattened f
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Restored_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
    OR TRIM(map.Assessable_Unit_ID) = TRIM(mast.Master_Numeric_ID)
ORDER BY f.Cleaned_Numeric_Amount DESC;